# ReMorph — Master Colab: staged GRPO → master charts → Hub

**Goal:** stronger tool policy via TRL GRPO (500 → 1500 → 3000 prompts), overlay **master** plots for judges, upload `artifacts/submission` to a HF Dataset.

**Runtime:** GPU (T4+). CPU will not finish GRPO in reasonable time.

**Repo:** `VedantPancholi/remorph-openenv-submission` (same as HF Jobs clone).

## 0. Hugging Face token (upload + model download)

In Colab: **Secrets** → add `HF_TOKEN` (read + write for datasets you upload to).

Or export in a prior cell: `import os; os.environ["HF_TOKEN"] = "hf_..."` (do not commit tokens).

In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab userdata")
except Exception:
    assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in the environment"
    print("HF_TOKEN from environment")

## 1. GPU check

In [ ]:
!nvidia-smi || true

## 2. Clone submission repo and install

Uses GPU-friendly `requirements-training.txt` (torch + trl from PyPI on Colab).

In [ ]:
%%bash
set -e
REPO_URL="${REPO_URL:-https://github.com/VedantPancholi/remorph-openenv-submission.git}"
REPO_DIR="${REPO_DIR:-remorph-openenv-submission}"
if [[ ! -d "$REPO_DIR" ]]; then git clone --depth 1 "$REPO_URL" "$REPO_DIR"; fi
cd "$REPO_DIR"
pip install -q -U pip
pip install -q -r requirements.txt
pip install -q -r requirements-training.txt
echo "Ready in $(pwd)"

## 3. Run full staged pipeline (datasets + v1/v2/v3 GRPO + master plots + upload)

Tune without editing the notebook:

- `MAX_STEPS_V1`, `MAX_STEPS_V2`, `MAX_STEPS_V3` — more steps = stronger fit, longer run
- `MODEL` — e.g. `Qwen/Qwen2.5-0.5B-Instruct` or larger if VRAM allows
- `SKIP_UPLOAD=1` — training only, no dataset repo upload
- `UPLOAD_PATH_IN_REPO` — folder name inside your HF dataset repo

**Iterating for hackathon wins:** after each run, inspect `artifacts/submission/trl_run_ledger.json` and `plots/master/*.png`; bump steps or LR and re-run with a new `UPLOAD_PATH_IN_REPO` (e.g. `run_2026_04_26_b`).

In [ ]:
%%bash
set -e
cd remorph-openenv-submission
export MODEL="${MODEL:-Qwen/Qwen2.5-0.5B-Instruct}"
export MAX_STEPS_V1="${MAX_STEPS_V1:-100}"
export MAX_STEPS_V2="${MAX_STEPS_V2:-200}"
export MAX_STEPS_V3="${MAX_STEPS_V3:-300}"
export HF_DATASET_REPO="${HF_DATASET_REPO:-Jenish31/remorph-training-artifacts}"
export UPLOAD_PATH_IN_REPO="${UPLOAD_PATH_IN_REPO:-colab_full_staged_run}"
export SKIP_UPLOAD="${SKIP_UPLOAD:-0}"
chmod +x scripts/hf_run_staged_grpo_full.sh
bash scripts/hf_run_staged_grpo_full.sh

## 4. (Optional) Zip artifacts for download

Use if you want a single file from Colab → local machine.

In [ ]:
%%bash
cd remorph-openenv-submission
zip -rq /content/remorph_submission_artifacts.zip artifacts/submission
ls -lh /content/remorph_submission_artifacts.zip